# Chapter 8 — Exercise Solutions## RL for HPO & AutoML[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

### Solution 8.1: Bayesian Optimisation Baseline

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from scipy.stats import norm

X, y = load_breast_cancer(return_X_y=True)
X = StandardScaler().fit_transform(X)

def evaluate_xgb(n_estimators=100, max_depth=6, learning_rate=0.1, subsample=0.8):
    try:
        from xgboost import XGBClassifier
        model = XGBClassifier(n_estimators=int(n_estimators), max_depth=int(max_depth),
                              learning_rate=learning_rate, subsample=subsample,
                              eval_metric='auc', verbosity=0, random_state=42)
        return cross_val_score(model, X, y, cv=3, scoring='roc_auc').mean()
    except ImportError:
        # Fallback: sklearn GradientBoosting
        from sklearn.ensemble import GradientBoostingClassifier
        model = GradientBoostingClassifier(n_estimators=int(n_estimators),
                                            max_depth=int(max_depth),
                                            learning_rate=learning_rate,
                                            subsample=subsample, random_state=42)
        return cross_val_score(model, X, y, cv=3, scoring='roc_auc').mean()

# Bayesian Optimisation with Gaussian Process
# Search space: normalised [0,1] for each hyperparameter
BOUNDS = np.array([[0,1]]*4)  # 4 hyperparameters, each in [0,1]

def denormalise(x):
    """Map [0,1]^4 → actual hyperparameter values."""
    return {
        'n_estimators': int(50  + x[0] * 450),   # 50-500
        'max_depth':    int(3   + x[1] * 7),      # 3-10
        'learning_rate': 0.01  + x[2] * 0.49,     # 0.01-0.5
        'subsample':     0.5   + x[3] * 0.5,      # 0.5-1.0
    }

def acquisition_ei(X_cand, gp, y_best, xi=0.01):
    """Expected Improvement acquisition function."""
    mu, sigma = gp.predict(X_cand, return_std=True)
    Z = (mu - y_best - xi) / (sigma + 1e-9)
    return (mu - y_best - xi) * norm.cdf(Z) + sigma * norm.pdf(Z)

# Random initial points
np.random.seed(42)
X_obs = np.random.rand(5, 4)
y_obs = np.array([evaluate_xgb(**denormalise(x)) for x in X_obs])
print(f"Initial 5 random evaluations: {y_obs.round(4)}")

# Bayesian Optimisation loop
gp = GaussianProcessRegressor(kernel=Matern(nu=2.5), normalize_y=True, random_state=42)
best_auc = y_obs.max()

for i in range(10):
    gp.fit(X_obs, y_obs)
    candidates = np.random.rand(1000, 4)
    ei = acquisition_ei(candidates, gp, best_auc)
    next_x = candidates[ei.argmax()]
    next_y = evaluate_xgb(**denormalise(next_x))
    X_obs = np.vstack([X_obs, next_x])
    y_obs = np.append(y_obs, next_y)
    if next_y > best_auc:
        best_auc = next_y
    print(f"BO iter {i+1:2d}: AUC={next_y:.4f} (best={best_auc:.4f})")

print(f"\nFinal best AUC: {best_auc:.4f}")
print("Expected: BO typically beats random search by iteration 8-12")

# YOUR TURN: Compare BO vs random search on the same evaluation budget (15 calls)
# Which finds better hyperparameters? Which is more consistent across random seeds?


### Solution 8.2: Successive Halving

In [ ]:
import numpy as np

def successive_halving(param_configs, eval_fn, min_budget=1, max_budget=27, eta=3):
    """
    Successive Halving: run all configs for min_budget, keep top 1/eta,
    double budget, repeat until one config remains.
    eta=3: halve to 1/3 each round.
    """
    n = len(param_configs)
    budget = min_budget
    results_history = []

    print(f"Starting with {n} configs, min_budget={min_budget}")
    configs = param_configs.copy()

    while len(configs) > 1:
        # Evaluate all remaining configs at current budget
        scores = []
        for cfg in configs:
            # budget = number of CV folds (proxy for compute)
            auc = eval_fn(cv_folds=max(1, int(budget)), **cfg)
            scores.append(auc)

        # Keep top 1/eta configs
        scores = np.array(scores)
        n_keep = max(1, len(configs) // eta)
        top_indices = np.argsort(scores)[-n_keep:]
        configs = [configs[i] for i in top_indices]
        results_history.append({
            'budget': budget, 'n_configs': len(configs)+n_keep,
            'best_score': scores.max(), 'kept': n_keep
        })
        print(f"  Budget={budget:3.0f}: best={scores.max():.4f}, kept {n_keep}/{len(scores)}")
        budget *= eta

    final_score = eval_fn(cv_folds=5, **configs[0])
    print(f"Winner config: {configs[0]}, Final AUC: {final_score:.4f}")
    return configs[0], final_score, results_history

# Proxy evaluation function (faster than real cross-validation)
np.random.seed(42)
def fast_eval(cv_folds=3, n_estimators=100, max_depth=6, learning_rate=0.1, subsample=0.8):
    """Simulated evaluation with noise inversely proportional to budget."""
    true_quality = (n_estimators/500)*0.1 + (1/max_depth)*0.05 + (1-learning_rate)*0.1
    noise = np.random.normal(0, 0.05/np.sqrt(cv_folds))
    return min(0.99, max(0.5, 0.82 + true_quality + noise))

# Generate random configs
configs = [{'n_estimators': np.random.randint(50,500),
            'max_depth': np.random.randint(3,10),
            'learning_rate': np.random.uniform(0.01,0.5),
            'subsample': np.random.uniform(0.5,1.0)} for _ in range(27)]

best_cfg, best_score, history = successive_halving(configs, fast_eval)
print(f"\nSuccessive Halving total function evaluations: ~{sum(h['n_configs'] for h in history)}")
print("vs Random Search at same budget: random would evaluate each config once")

# YOUR TURN: Implement Hyperband (multiple Successive Halving brackets)
# Hyperband runs SH with different initial budgets to handle different convergence speeds
